# B-02｜rPPG 真實影片 / 即時攝影機分析

**用法：改下面那一格，然後從上到下全部執行（Kernel → Restart & Run All，
或每一格按 Shift+Enter）。**

支援兩種來源：
- `SOURCE = "file"`：分析 `data/recordings/` 裡的 mp4，適合錄完一支就檢查一次
- `SOURCE = "camera"`：即時攝影機分析，錄製時螢幕會跳出一個獨立預覽視窗

**為什麼即時攝影機這個模式重要**：實際的 eKYC 開戶流程裡，系統面對的永遠是
即時攝影機串流，不是預先錄好的檔案——檔案只是我們開發階段方便重複測試用的
替代品。跑通攝影機這條路徑，才是真正貼近上線後的樣子。

跑不出正確心率時的排查順序：
1. 先看 `b_01_rppg_synthetic_test.ipynb` 是不是還正常 → 正常表示訊號處理沒壞
2. 看第 3 節的 ROI 疊圖 → ROI 框錯的話數字全是假的，而且不會報錯
3. 看第 1 節的幀率檢查 → fps 算錯會讓心率整個算錯

## ⚙️ 只要改這一格

In [ ]:
SOURCE = "camera"   # "file" 或 "camera"

# SOURCE = "file" 時才用得到
VIDEO = "data/recordings/real_normal_001.mp4"

# SOURCE = "camera" 時才用得到
CAMERA_INDEX = 0        # 0 = 預設攝影機。OBS Virtual Camera 通常是 1 或 2，
                        # 抓不到人臉的話換個數字試試
RECORD_SECONDS = 20.0   # 錄製秒數，不建議低於 config.VIDEO_SECONDS_MIN（15 秒）

# camera 模式錄完會不會存成檔案。不存的話，錄的東西只活在這個 kernel 的
# 記憶體裡，關掉 Jupyter 或重跑這一格就會消失，找不回來。
SAVE_RECORDING = True
# 條件標籤，照 CONVENTIONS.md §7.1：normal / dark / bright / near / far / moving / glasses
SAVE_CONDITION = "normal"

# 兩種來源都要填
GROUND_TRUTH_BPM = 68.0   # 手錶/手機同步量到的心率，沒量的話填 None
SCALE = 0.4                # 縮放比例，縮小可省記憶體、順便壓掉一點雜訊

In [ ]:
import sys
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import font_manager

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import config
from image_utils import quality
from track2_rppg import analyzer
from track2_rppg import signal_utils as su

available = {f.name for f in font_manager.fontManager.ttflist}
for c in ("Microsoft JhengHei", "Microsoft YaHei", "PingFang TC", "Noto Sans CJK TC", "SimHei"):
    if c in available:
        plt.rcParams["font.sans-serif"] = [c, "DejaVu Sans"]
        break
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110

assert SOURCE in ("file", "camera"), 'SOURCE 只能是 "file" 或 "camera"'
print(f"來源模式：{SOURCE}")

## 1. 取得影格

**這一格是唯一因來源不同而分岔的地方**，之後第 2 節開始不管資料是從檔案
還是攝影機來的，用的都是同一組 `frames` 和 `FPS` 變數，後面每一格完全不用改。

### file 模式：幀率檢查

**最重要的是抖動比。** 手機常錄成可變幀率（VFR），影格間隔不等長，
頻率會整個算錯——20 秒的影片可能算出兩倍或一半的心率。
抖動比 < 0.05 才算固定幀率（CFR）。

### camera 模式：不能信任攝影機宣告的 fps

`cv2.CAP_PROP_FPS` 拿到的是攝影機**宣告**的理論值，但實際擷取速率會隨
光線與 CPU 負載漂移——**光線不足時攝影機會自動延長曝光，fps 可能從 30 掉到 15**。
照宣告值去算頻譜，心率會直接算成兩倍或算錯。

解法是自己記錄每一格被讀到的實際時刻，用「總格數 ÷ 總秒數」算出真正的 fps。
下面這格就是這樣做的。

**執行到這一格時，螢幕會跳出一個獨立的預覽視窗**（不在 Jupyter 頁面裡，是一個
另外彈出的小視窗），倒數 3 秒後開始錄。錄製時請：
- 正對鏡頭，距離約 50cm
- **房間盡量調亮**——這是影響結果最大的單一因素
- 盡量不要說話、不要大動作，微笑不用憋著但不要一直換表情

錄完會自動關閉視窗；也可以按視窗裡的 `q` 鍵提前結束。

In [ ]:
SAVED_VIDEO_PATH = None   # 只有 camera 模式 + SAVE_RECORDING 才會有值

if SOURCE == "file":
    VIDEO_PATH = ROOT / VIDEO
    assert VIDEO_PATH.exists(), f"找不到影片：{VIDEO_PATH}"
    print(f"影片：{VIDEO_PATH}")
    print(f"大小：{VIDEO_PATH.stat().st_size / 1024**2:.1f} MB\n")

    cap = cv2.VideoCapture(str(VIDEO_PATH))
    assert cap.isOpened(), "OpenCV 打不開這個檔案"

    declared_fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    stamps = []
    while cap.grab():
        stamps.append(cap.get(cv2.CAP_PROP_POS_MSEC))

    stamps = np.array(stamps)
    duration = stamps[-1] / 1000
    measured_fps = len(stamps) / duration
    deltas = np.diff(stamps)
    deltas = deltas[deltas > 0]
    jitter = deltas.std() / np.median(deltas) if len(deltas) else 0.0

    print(f"解析度    : {w} x {h}")
    print(f"宣告 fps  : {declared_fps:.3f}")
    print(f"實測 fps  : {measured_fps:.3f}")
    print(f"影格數    : {len(stamps)}")
    print(f"長度      : {duration:.2f} 秒")
    print(f"抖動比    : {jitter:.3f}  ->  "
          f"{'固定幀率 CFR' if jitter < 0.05 else '⚠️ 可變幀率 VFR，心率可能算錯'}")

    # 重新從頭讀一次，這次真正載入影格（BGR 轉 RGB、依 SCALE 縮小）
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    frames = []
    while True:
        ok, bgr = cap.read()
        if not ok:
            break
        small = cv2.resize(bgr, None, fx=SCALE, fy=SCALE, interpolation=cv2.INTER_AREA)
        frames.append(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))
    cap.release()

    FPS = measured_fps
    if duration < config.VIDEO_SECONDS_MIN:
        print(f"\n⚠️ 長度不足 {config.VIDEO_SECONDS_MIN} 秒，頻譜解析度不夠，建議重錄")

else:  # SOURCE == "camera"
    cap = cv2.VideoCapture(CAMERA_INDEX)
    assert cap.isOpened(), f"打不開攝影機 index={CAMERA_INDEX}，換個數字試試（0/1/2）"

    declared_fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"攝影機 index={CAMERA_INDEX} 已開啟（宣告 fps={declared_fps:.1f}）")
    print("3 秒後開始錄製，請正對鏡頭、房間調亮、盡量不要動...")
    time.sleep(3)
    print(f"開始錄製 {RECORD_SECONDS:.0f} 秒...（預覽視窗按 q 可提前結束）")

    frames = []
    raw_timestamps = []
    t_start = time.time()
    while True:
        elapsed = time.time() - t_start
        if elapsed >= RECORD_SECONDS:
            break
        ok, bgr = cap.read()
        if not ok:
            print("讀取影格失敗，提前結束")
            break

        raw_timestamps.append(elapsed)
        # 分析用的 frames 不鏡像，跟其他地方（mp4 檔案讀進來）保持一致，
        # 左頰/右頰的 ROI 命名才不會因為鏡像而錯位
        small = cv2.resize(bgr, None, fx=SCALE, fy=SCALE, interpolation=cv2.INTER_AREA)
        frames.append(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))

        # 預覽視窗鏡像顯示（左右翻轉），只影響你看到的畫面，方便對鏡頭定位——
        # 未鏡像的攝影機畫面是「別人看你的角度」，鏡像後才是你熟悉的「照鏡子」視角
        preview = cv2.flip(bgr, 1)
        cv2.putText(preview, f"{elapsed:.1f}s / {RECORD_SECONDS:.0f}s", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.imshow("rPPG 錄製中（按 q 提前結束，鏡像顯示不影響分析）", preview)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            print("使用者提前結束")
            break

    cap.release()
    cv2.destroyAllWindows()

    raw_timestamps = np.array(raw_timestamps)
    duration = raw_timestamps[-1]
    FPS = len(frames) / duration          # ← 實測 fps，不是宣告值
    deltas = np.diff(raw_timestamps)
    jitter = deltas.std() / np.median(deltas) if len(deltas) else 0.0

    print(f"\n擷取完成：{len(frames)} 格，{duration:.1f} 秒")
    print(f"攝影機宣告 fps：{declared_fps:.2f}")
    print(f"實際擷取 fps  ：{FPS:.2f}   <- 分析用這個，不是宣告值")
    print(f"影格間隔抖動  ：{jitter:.3f}")
    if declared_fps > 0 and abs(FPS - declared_fps) > 3:
        print(f"⚠️ 實際與宣告差 {abs(FPS - declared_fps):.1f} fps，"
              f"用宣告值會讓心率算錯 {declared_fps / FPS:.2f} 倍")
    if duration < config.VIDEO_SECONDS_MIN:
        print(f"\n⚠️ 長度不足 {config.VIDEO_SECONDS_MIN} 秒，頻譜解析度不夠")

    # 存成 mp4，照 CONVENTIONS.md §7.1 命名：real_{condition}_{三位編號}.mp4
    # 沒有這一步的話，剛才錄的東西只活在這個 kernel 的記憶體裡，
    # 關掉 Jupyter 或重跑這一格就會消失，找不回來。
    if SAVE_RECORDING:
        rec_dir = ROOT / "data" / "recordings"
        rec_dir.mkdir(parents=True, exist_ok=True)
        n = 1
        while (rec_dir / f"real_{SAVE_CONDITION}_{n:03d}.mp4").exists():
            n += 1
        SAVED_VIDEO_PATH = rec_dir / f"real_{SAVE_CONDITION}_{n:03d}.mp4"

        h, w = frames[0].shape[:2]
        writer = cv2.VideoWriter(str(SAVED_VIDEO_PATH), cv2.VideoWriter_fourcc(*"mp4v"), FPS, (w, h))
        for f in frames:
            writer.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))   # 存檔要 BGR
        writer.release()
        print(f"\n已存檔：{SAVED_VIDEO_PATH.relative_to(ROOT)}")
        print("（第 7 節分析完後會在同資料夾補一份對應的 .json，記錄實測數據）")

mem_mb = len(frames) * frames[0].nbytes / 1024**2
print(f"\n共 {len(frames)} 格，尺寸 {frames[0].shape}，佔用約 {mem_mb:.0f} MB，分析用 FPS={FPS:.2f}")

plt.figure(figsize=(4, 6))
plt.imshow(frames[len(frames)//2])
plt.axis("off")
plt.title("中間一格")
plt.show()

## 2. 影像品質檢查

四項指標對照 `config.py` 的門檻。任何一項不過，`passed` 就是 False。

**清晰度（blurScore）特別重要**：實測真人實拍約 900–1500，
螢幕翻拍只有 50–200 左右，是分辨真人與翻拍最有力的單一指標。

In [ ]:
q = quality.check_image_quality(frames)

rows = [
    ("清晰度",   q["blurScore"],        f">= {config.QUALITY_BLUR_MIN:.0f}",
     q["blurScore"] >= config.QUALITY_BLUR_MIN),
    ("亮度",     q["brightness"],       f"{config.QUALITY_BRIGHTNESS_MIN} ~ {config.QUALITY_BRIGHTNESS_MAX}",
     config.QUALITY_BRIGHTNESS_MIN <= q["brightness"] <= config.QUALITY_BRIGHTNESS_MAX),
    ("對比度",   q["contrast"],         f">= {config.QUALITY_CONTRAST_MIN:.0f}",
     q["contrast"] >= config.QUALITY_CONTRAST_MIN),
    ("過曝比例", q["overexposedRatio"], f"<= {config.QUALITY_OVEREXPOSED_MAX:.2f}",
     q["overexposedRatio"] <= config.QUALITY_OVEREXPOSED_MAX),
    ("臉部佔比", q["faceRatio"],        f">= {config.QUALITY_FACE_RATIO_MIN:.2f}",
     q["faceRatio"] >= config.QUALITY_FACE_RATIO_MIN),
]

print(f"{'指標':<10}{'數值':>10}   {'門檻':<14}")
print("-" * 46)
for name, val, thr, ok in rows:
    print(f"{name:<10}{val:>10.3f}   {thr:<14} {'OK' if ok else '<-- 不通過'}")
print("-" * 46)
print(f"整體：{'通過' if q['passed'] else '不通過'}")
if q["message"]:
    print(f"原因：{q['message']}")
print()
print("註：faceRatio 需要 InsightFace，第一次執行會下載約 300MB 模型。")

## 3. ROI 疊圖｜**這一步不能跳過**

綠=額頭、紅=左頰、紫=右頰。

ROI 框錯位置時**程式不會報錯，但算出來的數字全是假的**，
而且很難從數字看出問題。每次都要親眼確認一次。

常見問題：
- **瀏海壓到額頭 ROI** → 用髮夾把瀏海往上固定
- 戴眼鏡的反光落在臉頰 ROI → 調整角度
- 臉太偏，某一頰跑出畫面

In [ ]:
landmarks_list = analyzer.extract_landmarks(frames, FPS)
found = sum(l is not None for l in landmarks_list)
rate = found / len(frames)

print(f"偵測到臉：{found}/{len(frames)} 格（{rate:.1%}）")
if rate < config.RPPG_MIN_FACE_RATIO:
    print(f"⚠️ 偵測率低於 {config.RPPG_MIN_FACE_RATIO:.0%}，analyze_rppg 會直接回傳 detected=False")

idx = next((i for i, l in enumerate(landmarks_list) if l is not None), None)
assert idx is not None, "整支影片都沒偵測到臉，檢查影片是否太暗或臉未入鏡"

pts = landmarks_list[idx]
face_w = pts[:, 0].max() - pts[:, 0].min()
print(f"臉部寬度：{face_w:.0f} px  {'（偏小，建議調高 SCALE）' if face_w < 150 else 'OK'}")

overlay = analyzer.draw_roi_overlay(frames[idx], pts)
pad = 30
x0, x1 = int(pts[:,0].min()-pad), int(pts[:,0].max()+pad)
y0, y1 = int(pts[:,1].min()-pad), int(pts[:,1].max()+pad)
crop = overlay[max(0,y0):y1, max(0,x0):x1]

fig, axes = plt.subplots(1, 2, figsize=(9, 5))
axes[0].imshow(overlay); axes[0].set_title("全畫面"); axes[0].axis("off")
axes[1].imshow(crop);    axes[1].set_title("臉部放大｜綠=額頭 紅=左頰 紫=右頰"); axes[1].axis("off")
plt.tight_layout(); plt.show()

## 4. 心率分析

`analyze_rppg()` 的完整輸出，格式即 CONVENTIONS.md §4.4 的契約。

**怎麼看這格的結果**：
- `detected`：三項判定 a、b、c 是否**全部**通過。目前真人實拍的 SNR 常常過不了
  門檻（這是門檻待校準，不是你的問題，見第 8 節的參考值）
- `heartRate`：估算心率（bpm），就算 `detected=False` 也會給一個值，因為
  「主頻落在合理範圍」跟「訊噪比夠不夠」是分開判定的
- `snr`：訊號強度，> 0 表示訊號比雜訊強；真人實拍實測落在 −1 附近也算正常
- `roiConsistency`：0–1，三個 ROI 心率有多一致。這是三項指標裡對翻拍攻擊
  最穩定的一個（真人 0.74–0.81，翻拍 0.31 左右）

In [ ]:
t0 = time.time()
result = analyzer.analyze_rppg(frames, FPS)
elapsed = time.time() - t0

print(f"耗時 {elapsed:.1f} 秒（{len(frames)/elapsed:.0f} fps）\n")
print(f"detected       : {result['detected']}")
print(f"heartRate      : {result['heartRate']:.2f} bpm" if result['heartRate'] else "heartRate      : None")
print(f"snr            : {result['snr']:+.2f} dB   (門檻 {config.RPPG_SNR_MIN})")
print(f"roiConsistency : {result['roiConsistency']:.4f}  (門檻 {config.RPPG_ROI_CONSISTENCY_MIN})")
print(f"waveform 長度  : {len(result['waveform'])}")
print(f"spectrum 長度  : {len(result['spectrum'])}")
print()
print("三項判定：")
for c in result["checks"]:
    print(f"  [{'v' if c['passed'] else ' '}] {c['label']}")

## 5. 三個 ROI 分開看

`analyze_rppg` 只回傳一個代表值（取 SNR 最高的 ROI），
但三個 ROI 各自的數字才看得出問題出在哪。

下面那張波形圖：**三條線如果前幾秒看得出同相位起伏，就是抓到真實心跳**；
頻譜圖：**三條線的峰疊在同一個心率附近，是訊號真實的另一個佐證**。

In [ ]:
roi_detail = {}
for name, indices in analyzer.ROI_DEFINITIONS.items():
    sig = su.extract_roi_signal(frames, landmarks_list, indices)
    r = analyzer._analyze_single_roi(sig, FPS)
    if r:
        roi_detail[name] = r

labels = {"forehead": "額頭", "left_cheek": "左頰", "right_cheek": "右頰"}
colors = {"forehead": "#2ca02c", "left_cheek": "#d62728", "right_cheek": "#7c4dff"}

print(f"{'ROI':<8}{'心率 (bpm)':>12}{'SNR (dB)':>12}")
print("-" * 32)
for name, r in roi_detail.items():
    print(f"{labels[name]:<8}{r['heart_rate']:>12.2f}{r['snr']:>+12.2f}")
print("-" * 32)

if len(roi_detail) >= 2:
    hrs = [r["heart_rate"] for r in roi_detail.values()]
    print(f"{'跨度':<8}{max(hrs)-min(hrs):>12.2f}")
    print(f"\n真人參考：跨度 3.9-5.1 bpm｜翻拍參考：13.5-13.8 bpm")

In [ ]:
if roi_detail:
    fig, axes = plt.subplots(2, 1, figsize=(13, 7))
    t = np.arange(len(next(iter(roi_detail.values()))["filtered"])) / FPS

    for name, r in roi_detail.items():
        axes[0].plot(t, r["filtered"], lw=0.9, alpha=0.8, color=colors[name],
                     label=f"{labels[name]} {r['heart_rate']:.1f} bpm")
        axes[1].plot(r["freqs"] * 60, r["psd"] / r["psd"].max(), lw=1.3, color=colors[name],
                     label=f"{labels[name]} {r['heart_rate']:.1f} bpm / SNR {r['snr']:+.1f} dB")

    axes[0].set_xlim(0, min(10, t[-1]))
    axes[0].set_title("濾波後波形（前 10 秒）｜三條線同相位表示訊號真實")
    axes[0].set_xlabel("時間（秒）"); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

    axes[1].axvspan(config.RPPG_BAND_LOW*60, config.RPPG_BAND_HIGH*60, alpha=0.07, color="green")
    if GROUND_TRUTH_BPM:
        axes[1].axvline(GROUND_TRUTH_BPM, color="k", ls="--", lw=1.5,
                        label=f"實測 {GROUND_TRUTH_BPM:.0f} bpm")
    axes[1].set_xlim(30, 200)
    axes[1].set_title("功率頻譜｜三條線的峰重疊在同一處表示抓到真實心跳")
    axes[1].set_xlabel("心率（bpm）"); axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

    plt.tight_layout(); plt.show()

## 6. 對答案

In [ ]:
if GROUND_TRUTH_BPM and result["heartRate"]:
    err = abs(result["heartRate"] - GROUND_TRUTH_BPM)
    print(f"實測心率：{GROUND_TRUTH_BPM:.1f} bpm")
    print(f"估算心率：{result['heartRate']:.2f} bpm")
    print(f"誤差    ：{err:.2f} bpm   ->  {'✓ 達標（< 5 bpm）' if err < 5 else '✗ 超過 5 bpm'}")

    if roi_detail:
        errs = [abs(r["heart_rate"] - GROUND_TRUTH_BPM) for r in roi_detail.values()]
        print(f"\n三個 ROI 各自的誤差：{', '.join(f'{e:.2f}' for e in errs)} bpm")
        print(f"平均值的誤差：{abs(np.mean([r['heart_rate'] for r in roi_detail.values()]) - GROUND_TRUTH_BPM):.2f} bpm")
elif not GROUND_TRUTH_BPM:
    print("沒有填 GROUND_TRUTH_BPM，無法對答案。")
    print("下次錄影時記得同步用手錶量心率——沒有這個數字，這支影片就沒有驗證價值。")
else:
    print("沒有估出心率。看第 2 節的品質檢查與第 3 節的偵測率。")

## 7. 補存 .json（只有 camera 模式且有存檔時才會動作）

跟第 1 節存的 mp4 同名，記錄這次錄影的完整量測結果，格式比照
`data/recordings/real_normal_001.json` 那份。以後想核對某支影片當初
量出的數字，或整理 100 段自製資料的統計時，不用重新跑一次分析。

In [ ]:
import json

if SOURCE == "camera" and SAVED_VIDEO_PATH is not None:
    json_path = SAVED_VIDEO_PATH.with_suffix(".json")
    record = {
        "sampleType": "real",
        "condition": SAVE_CONDITION,
        "recordedAt": time.strftime("%Y-%m-%d %H:%M:%S"),
        "source": "live_camera",
        "groundTruthHeartRate": GROUND_TRUTH_BPM,
        "groundTruthSource": "手錶/手機，錄影當下同步量測" if GROUND_TRUTH_BPM else None,
        "recording": {
            "durationSec": round(float(duration), 2),
            "fps": round(float(FPS), 3),
            "totalFrames": len(frames),
            "cameraIndex": CAMERA_INDEX,
        },
        "measured": {
            "brightness": q["brightness"],
            "blurScore": q["blurScore"],
            "contrast": q["contrast"],
            "overexposedRatio": q["overexposedRatio"],
            "faceRatio": q["faceRatio"],
            "qualityPassed": q["passed"],
            "heartRate": result["heartRate"],
            "snr": result["snr"],
            "roiConsistency": result["roiConsistency"],
            "detected": result["detected"],
            "roiHeartRates": {name: r["heart_rate"] for name, r in roi_detail.items()},
            "roiSnr": {name: r["snr"] for name, r in roi_detail.items()},
        },
        "notes": "",
    }
    json_path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"已存：{json_path.relative_to(ROOT)}")
else:
    print("這次沒有存檔（SOURCE 不是 camera，或 SAVE_RECORDING 是 False）")

## 8. 這支能不能用／結果正不正常？

| 症狀 | 要重錄嗎 |
|---|---|
| 品質檢查 `passed=False` | **要** |
| 臉部偵測率 < 50% | **要** |
| 三個 ROI 跨度 > 20 bpm | **要**（訊號是雜訊，不是心跳） |
| 與手錶誤差 > 5 bpm | **要** |
| file 模式抖動比 > 0.05（可變幀率） | **要**，且要改手機設定 |
| `detected=False` 但心率算得準、跨度小 | **不用**——門檻待階段 3 校準 |
| SNR 是負的 | **不用**——真人實拍實測就是 −1 到 −1.4 dB |

**參考值**（2026-08-14 實測，手錶 68 bpm）：

| | 真人開燈 | 真人關燈 | 翻拍・螢幕 | 翻拍・手機 |
|---|---|---|---|---|
| 清晰度 | 1463 | 935 | 51 | 204 |
| 三 ROI 跨度 | 5.1 | 3.9 | 13.8 | 13.5 |
| roiConsistency | 0.744 | 0.807 | 0.308 | 0.323 |
| SNR | −1.09 | −1.38 | −3.04 | −2.92 |
| 與手錶誤差 | 1.70 | 1.85 | — | — |

**即時攝影機沒有 H.264/HEVC 壓縮**，理論上訊號品質應該比手機錄的 mp4 更好——
如果你這次用攝影機測出來的 SNR 明顯高於上表的真人參考值，這就是證據。